# Basline model
In this notebook, a Vision Transformer (ViT) is implemented as a powerful baseline model for the classification of birds. 
* The Google/vit-base-patch16-224 architecture was employed through the utilisation of the Hugging Face Transformers library.
* Transfer Learning: The pre-trained weights are loaded and the model is fine-tuned specifically on the dataset of 200 bird species.

### Imports

In [8]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import StratifiedKFold
from pathlib import Path
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter

# Hugging Face imports
from transformers import ViTForImageClassification

# Device setup
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
if torch.cuda.is_available(): device = torch.device("cuda")
print("Device:", device)

Device: mps


### Configs

In [ ]:
DATA_DIR = Path("''\train_images/train_images")
csv_path = Path("..\train_images.csv") 

batch_size = 32
# ViT finetunt beter met een lage learning rate
learning_rate = 2e-5 
weight_decay = 0.01
# Baseline training hoeft niet lang te duren met pre-trained weights
num_epochs = 10
num_classes = 200
seed = 42
model_name = 'google/vit-base-patch16-224'

torch.manual_seed(seed)
np.random.seed(seed)
print("Config loaded. Model:", model_name)

Config loaded. Model: google/vit-base-patch16-224


### Datapreperation

### Dataset class

In [3]:
class BirdDataset(Dataset):
    def __init__(self, csv_file, root_dir, img_col_idx, label_col_idx, transform=None):
        self.data = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.img_col_idx = img_col_idx
        self.label_col_idx = label_col_idx
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        filename = str(self.data.iloc[idx, self.img_col_idx])
        clean_filename = filename.lstrip('/').lstrip('\\')
        img_path = os.path.join(self.root_dir, clean_filename)

        try:
            image = Image.open(img_path).convert('RGB')
        except (FileNotFoundError, OSError):
            print(f"Could not open {img_path}, using black image.")
            image = Image.new('RGB', (224, 224), (0, 0, 0))

        # PyTorch labels: 0..199
        raw_label = int(self.data.iloc[idx, self.label_col_idx])
        label = raw_label - 1

        if self.transform:
            image = self.transform(image)

        return image, label

### Transformers

In [5]:
vit_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

### Train validation split

In [ ]:
full_train_dataset = BirdDataset('..\train_images.csv', '..\train_images', 0, 1, transform=vit_transform)
test_dataset = BirdDataset('..\test_images_path.csv', '..\test_images', 1, 2, transform=vit_transform)

labels = full_train_dataset.data.iloc[:, 1].values - 1

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
for train_idx, val_idx in skf.split(np.zeros(len(labels)), labels):
    train_dataset = Subset(full_train_dataset, train_idx)
    val_dataset   = Subset(full_train_dataset, val_idx)
    break

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")
print(f"Test set: {len(test_dataset)} images")

Train: 3140, Val: 786
Test set: 4000 images


### Model setup

In [7]:
print(f"Loading pre-trained model: {model_name}...")

model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=num_classes,
    ignore_mismatched_sizes=True
)

model.to(device)
print("ViT Model loaded!")

Loading pre-trained model: google/vit-base-patch16-224...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([200]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([200, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ViT Model loaded!


### Training + TensorBoard Logging

In [ ]:
writer = SummaryWriter(log_dir="..\runs/baseline_vit")

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
criterion = nn.CrossEntropyLoss()

best_acc = 0

for epoch in range(1, num_epochs+1):
    model.train()
    total_loss, correct, total = 0, 0, 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs} Training", leave=False)
    for batch_idx, (imgs, labels) in enumerate(loop):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()

        # HuggingFace model output
        outputs = model(imgs)
        logits = outputs.logits
        
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        
        # Predictions
        _, preds = torch.max(logits, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        loop.set_postfix(loss=loss.item(), acc=(correct/total))

    train_acc = correct / total
    train_loss = total_loss / total

    # Validation
    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            logits = outputs.logits
            
            loss = criterion(logits, labels)
            val_loss += loss.item() * imgs.size(0)
            
            _, preds = torch.max(logits, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    val_loss = val_loss / total

    print(f"Epoch {epoch}/{num_epochs} | "
          f"Train Acc: {train_acc:.4f} | "
          f"Val Acc: {val_acc:.4f} | Val Loss: {val_loss:.4f}")

    # logging to Tensorboard
    writer.add_scalar("Accuracy/train", train_acc, epoch)
    writer.add_scalar("Accuracy/val", val_acc, epoch)
    writer.add_scalar("Loss/train", train_loss, epoch)
    writer.add_scalar("Loss/val", val_loss, epoch)

    # Save best model
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "..\Models\baseline_vit_model.pth")
        print("New best baseline model saved")


writer.close()

Epoch 1/10 | Train Acc: 0.9997 | Val Acc: 0.7570 | Val Loss: 1.4399
New best baseline model saved


Epoch 2/10 | Train Acc: 0.9997 | Val Acc: 0.7621 | Val Loss: 1.3671
New best baseline model saved


Epoch 3/10 | Train Acc: 1.0000 | Val Acc: 0.7608 | Val Loss: 1.3140


Epoch 4/10 | Train Acc: 1.0000 | Val Acc: 0.7583 | Val Loss: 1.2790


Epoch 5/10 | Train Acc: 1.0000 | Val Acc: 0.7621 | Val Loss: 1.2368


Epoch 6/10 | Train Acc: 1.0000 | Val Acc: 0.7595 | Val Loss: 1.2151


Epoch 7/10 | Train Acc: 1.0000 | Val Acc: 0.7621 | Val Loss: 1.1952


Epoch 8/10 | Train Acc: 1.0000 | Val Acc: 0.7659 | Val Loss: 1.1837
New best baseline model saved


KeyboardInterrupt: 

### Test and submission 


In [ ]:
# Load best model
checkpoint_path = "..\Models\baseline_vit_model.pth"
if os.path.exists(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print(f"Loaded weights from {checkpoint_path}")

model.eval()
predictions = []

print("Starting baseline predictions...")
with torch.no_grad():
    for inputs, _ in tqdm(test_loader, desc="Predicting"):
        inputs = inputs.to(device)
        outputs = model(inputs)
        logits = outputs.logits
        
        _, preds = torch.max(logits, 1)
        predictions.extend((preds.cpu().numpy() + 1))

# Save as baseline.csv 
submission = pd.DataFrame({
    "id": range(1, len(predictions) + 1),
    "label": predictions
})

output_file = "..\Submission_csv\baseline.csv"
submission.to_csv(output_file, index=False)
print(f"Submission ready: {output_file}")
print(submission.head())

Loaded weights from baseline_vit_model.pth
Starting baseline predictions...


Predicting: 100%|██████████| 125/125 [03:24<00:00,  1.64s/it]

Submission ready: baseline.csv
   id  label
0   1     67
1   2     38
2   3     74
3   4     12
4   5     74
